# Description
### EEG signals
Electroencephalography helps to acquire brain signals from various parts of the brain from scalp to surface area. These signal are categorized as **delta**, **theta**, **alpha**, **beta**, and **gamma**.

### Experiment
EEG signal data from 10 college students was collected while they watched MOOC videos. There were twenty videos, 10 were on basic topics like algebra and the other 10 were complex like Quantum Mechanics.

### Procedures
The students wore a single channel wireless headset, to capture their eeg signals and reported their confusion on a scale of one to seven, one being easiest and 7 being tough.

# Important Points to note (mentioned in discussions by the author)
#### Participant #3 and #7 are outliers and can cause issues.
#### 0 values for **Attention** and **Mediation**

In [ ]:
import numpy as np
import pandas
import seaborn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
eeg_df = pandas.read_csv('../input/confused-eeg/EEG_data.csv')

In [ ]:
eeg_df.head()

In [ ]:
print(eeg_df.info())

# Add demographic data

In [ ]:
dem_df = pandas.read_csv('../input/confused-eeg/demographic_info.csv')

In [ ]:
dem_df = dem_df.rename(columns = {'subject ID' : 'SubjectID'})

In [ ]:
dem_df['SubjectID'] = dem_df['SubjectID'].astype(np.float64)

In [ ]:
dem_df

In [ ]:
eeg_df = eeg_df.merge(dem_df, how = 'inner', on = 'SubjectID')

# One hot encoding

In [ ]:
eeg_df = pandas.get_dummies(eeg_df)

In [ ]:
from tabulate import tabulate
info = [[col, eeg_df[col].count(), eeg_df[col].max(), eeg_df[col].min()] for col in eeg_df.columns]
print(tabulate(info, headers = ['Feature', 'Count', 'Max', 'Min'], tablefmt = 'orgtbl'))

In [ ]:
print('Number of missing values : ' + str(eeg_df.isna().sum().sum()))

**Observations** : Mediation and Attention have some **0 value which is an error**, as claimed by the author in one of the discussions. Also the confusion is scaled between 0 and 1.


# Data Cleaning
It is obvious that **SubjectID** and **VideoID** will overfit the model, because there are 10 clips in all for each of the 10 students and these 60sec clips are divided into parts of 0.5sec samples. So, model may end up learning the IDs and predict on that basis.

In [ ]:
eeg_df = eeg_df.drop(['SubjectID', 'VideoID', 'predefinedlabel', ' gender_F'], axis = 1)

# Data Visualization

In [ ]:
eeg_df = eeg_df[eeg_df['Attention'] > 0.0]
eeg_df = eeg_df[eeg_df['Mediation'] > 0.0]

In [ ]:
eeg_df.hist(figsize = (15, 15))
plt.show()

In [ ]:
plt.figure(figsize = (15,15))
mat = eeg_df.corr()
seaborn.heatmap(mat, vmin = -1.0, square = True, annot = True)

There is a good correlation between **Gamma1** and **Beta2**, so let's look at their graphs.

In [ ]:
seaborn.pairplot(eeg_df.drop([col for col in eeg_df.columns if col != 'Gamma1' and col != 'Beta2'],axis=1))
plt.show()

This graph visually shows the correlation between **Gamma1** and **Beta2**. The correlation no longer looks promising enough to discard a column. So all columns will be used.

In [ ]:
eeg_df['user-definedlabeln'].unique()

In [ ]:
info = [[col, eeg_df[col].count(), eeg_df[col].max(), eeg_df[col].min()] for col in eeg_df.columns]
print(tabulate(info, headers = ['Feature', 'Count', 'Max', 'Min'], tablefmt = 'orgtbl'))

# Get the arrays from dataset

In [ ]:
X = np.array(eeg_df.drop(['user-definedlabeln'], axis = 1))
y = np.array(eeg_df['user-definedlabeln'])

In [ ]:
print(X.min())
print(X.max())

In [ ]:
print(y.min())
print(y.max())

# Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [ ]:
print(X.min())
print(X.max())

In [ ]:
print(y.min())
print(y.max())

In [ ]:
print(X.shape)
print(y.shape)

# Split into training and testing set

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
print(X_test.shape)
print(y_test.shape)

# Decision Forests and Boosting algorithms

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

model = ExtraTreesClassifier(n_estimators = 250, criterion = 'gini', min_samples_split = 2, max_features = None)
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

In [ ]:
print('Accuracy : ' + str(accuracy_score(y_test, pred)))
print(classification_report(y_test, pred))

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(y_test, pred)

In [ ]:
temp = pandas.DataFrame({'Not Confused' : mat[0,:], 'Confused' : mat[1,:]})

In [ ]:
plt.figure(figsize = (10,10))
seaborn.heatmap(temp, vmin = 0, annot = True, square = True)

# ANN Model

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input
from keras.layers import Dropout
from keras.layers import BatchNormalization

In [ ]:
def dens_layer (hiddenx) :

    model = Sequential()

    model.add(Dense(hiddenx, activation = 'relu', kernel_regularizer = 'l2'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))

    return model

In [ ]:
def ann (hidden1, hidden2,hidden3) :

    model = Sequential()

    model.add(Input(shape= (16,)))
    model.add(dens_layer(hidden1))
    model.add(dens_layer(hidden2))
    model.add(dens_layer(hidden3))

    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

    return model

In [ ]:
model = ann(32, 16, 16)
model.summary()

In [ ]:
from keras.callbacks import ModelCheckpoint
from keras.callbacks import ReduceLROnPlateau
reduce = ReduceLROnPlateau(monitor = 'val_loss', patience = 10, verbose = 1)
checkp = ModelCheckpoint('./result_model.h5', monitor = 'val_loss', save_best_only = True, verbose = 1)

In [ ]:
history = model.fit(X_train, y_train, batch_size = 32, epochs = 250, callbacks = [checkp, reduce], validation_data = (X_test, y_test))

In [ ]:
plt.figure(figsize = (20,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy (training and validation) vs Epochs')
plt.legend(['training accuracy' , 'validation accuracy'])
plt.xlabel('Epochs')
plt.ylabel('acccuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss (training and validation) vs Epochs')
plt.legend(['training loss', 'validation loss'])
plt.xlabel('Epochs')
plt.ylabel('losses')

In [ ]:
from keras.models import load_model
model = load_model('./result_model.h5')

In [ ]:
pred = model.predict(X_test)

In [ ]:
pred = pred.reshape(-1)

In [ ]:
pred = np.around(pred)

In [ ]:
print(pred[:10])
print(y_test[:10])

In [ ]:
print('Accuracy : ' + str(accuracy_score(y_test, pred)))
print(classification_report(y_test, pred))

In [ ]:
mat = confusion_matrix(y_test, pred)
temp = pandas.DataFrame({'Not Confused' : mat[0,:], 'Confused' : mat[1,:]})

In [ ]:
plt.figure(figsize = (10,10))
seaborn.heatmap(temp, vmin = 0.0, square = True, annot = True)